# Malvos — Fine-Tune DeepSeek-R1-Distill-Qwen-32B
### Built by Malvos Lab | Autonomous AI Coding Agent

**GPU:** A100 80GB — Runtime → Change runtime type → A100

## Cell 1: Install Dependencies

In [ ]:
%%capture
!pip install --no-deps unsloth "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets huggingface_hub

## Cell 2: Login to Hugging Face

In [ ]:
from huggingface_hub import login

HF_TOKEN    = "YOUR_HF_WRITE_TOKEN"  # hf_...
HF_USERNAME = "SHIKARI2"

login(token=HF_TOKEN)
print("Logged in!")

## Cell 3: Upload Dataset to Hugging Face Hub
Upload `dataset.jsonl` via the 📁 file panel on the left, then run this cell.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Create dataset repo
api.create_repo(
    repo_id   = f"{HF_USERNAME}/malvos-dataset",
    repo_type = "dataset",
    exist_ok  = True,
    token     = HF_TOKEN
)

# Push dataset file
api.upload_file(
    path_or_fileobj = "dataset.jsonl",
    path_in_repo    = "dataset.jsonl",
    repo_id         = f"{HF_USERNAME}/malvos-dataset",
    repo_type       = "dataset",
    token           = HF_TOKEN
)

print(f"Dataset pushed to: https://huggingface.co/datasets/{HF_USERNAME}/malvos-dataset")

## Cell 4: Load DeepSeek-R1-Distill-Qwen-32B (4-Bit QLoRA)

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 4096
dtype          = None
load_in_4bit   = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/DeepSeek-R1-Distill-Qwen-32B-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
)

print("Model loaded!")

## Cell 5: Attach LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha               = 32,
    lora_dropout             = 0,
    bias                     = "none",
    use_gradient_checkpointing = "unsloth",
    random_state             = 3407,
)

print("LoRA adapters attached!")

## Cell 6: Pull Dataset from Hugging Face & Format

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Apply ChatML template (DeepSeek-R1 / Qwen compatible)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    mapping       = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts  = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

# Pull from Hugging Face Hub
dataset = load_dataset(
    f"{HF_USERNAME}/malvos-dataset",
    split = "train",
    token = HF_TOKEN
)
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Loaded {len(dataset)} training examples from HF Hub!")
print("\nSample:\n", dataset[0]["text"][:500])

## Cell 7: Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = max_seq_length,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 10,
        max_steps                   = 150,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 1,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 3407,
        output_dir                  = "outputs",
    ),
)

print("Training started...")
trainer.train()
print("Training complete!")

## Cell 8: Test Malvos

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "You are Malvos — an elite autonomous AI coding agent built by Malvos Lab. "
    "You are NOT ChatGPT, Claude, or Copilot. "
    "You specialize in backend architecture, legacy migrations, private deployments, "
    "and micro-SaaS shipping. Jump straight into the solution with zero conversational filler."
)

test_prompts = [
    "Who are you?",
    "Are you ChatGPT?",
    "Write a Python async rate limiter using Redis with a token bucket algorithm.",
]

for prompt in test_prompts:
    print(f"\n{'='*60}\nUSER: {prompt}\n{'='*60}")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids      = inputs,
        max_new_tokens = 1024,
        use_cache      = True,
        temperature    = 0.6,
        top_p          = 0.95
    )
    print("Malvos:", tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

## Cell 9: Merge & Push to Hugging Face Hub

In [ ]:
NEW_MODEL_NAME = f"{HF_USERNAME}/Malvos-32B"

# Push full merged 16-bit weights (ready for vLLM, HF Endpoints, Ollama)
model.push_to_hub_merged(
    NEW_MODEL_NAME,
    tokenizer,
    save_method = "merged_16bit",
    token       = HF_TOKEN
)

print(f"Model live at: https://huggingface.co/{NEW_MODEL_NAME}")

## (Optional) GGUF for LM Studio / Ollama

In [ ]:
model.push_to_hub_gguf(
    f"{NEW_MODEL_NAME}-GGUF",
    tokenizer,
    quantization_method = "q4_k_m",
    token               = HF_TOKEN
)

print(f"GGUF pushed to: https://huggingface.co/{NEW_MODEL_NAME}-GGUF")